# Drive Me Crazy — Dashboard

The Looker Studio deliverable, done locally: an interactive view (plotly — hover, zoom, toggle series
by clicking the legend) of everything the two model notebooks wrote to `results/`. Re-run this notebook
after either of them to refresh.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

RESULTS = Path("results")
metrics = pd.concat([pd.read_csv(f) for f in sorted(RESULTS.glob("metrics_*.csv"))], ignore_index=True)
ORDER = [m for m in ["Persistence", "HistoricalAverage", "Ridge", "PDFormer-noDelay", "PDFormer"] if m in set(metrics.model)]
avg = metrics.query("horizon == 'avg'")
avg.pivot(index="model", columns="dataset", values="MAE").reindex(ORDER).round(2)

## 1. Model comparison — average error over the forecast horizon

In [ ]:
long = avg.melt(id_vars=["dataset", "model"], value_vars=["MAE", "RMSE", "MAPE"], var_name="metric")
fig = px.bar(long, x="dataset", y="value", color="model", facet_col="metric", barmode="group",
             category_orders={"model": ORDER}, title="Test-set error by dataset and model (lower is better)")
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

## 2. How error grows with the forecast horizon

In [ ]:
per_h = metrics.query("horizon != 'avg'").astype({"horizon": int})
fig = px.line(per_h, x="horizon", y="MAE", color="model", facet_col="dataset", markers=True,
              category_orders={"model": ORDER}, title="MAE by steps ahead (5-min steps for PeMS, 30-min for NYCTaxi)")
fig.update_yaxes(matches=None, showticklabels=True); fig.update_xaxes(matches=None)
fig.show()

## 3. Predictions vs. reality — first two test days, three sensors per dataset

In [ ]:
frames = []
for ds in sorted({f.name.split("_")[2] for f in RESULTS.glob("preds_sample_*_*.csv")}):
    parts = [pd.read_csv(f, parse_dates=["time_h1", "time_hL"]) for f in sorted(RESULTS.glob(f"preds_sample_{ds}_*.csv"))]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p.drop(columns=[c for c in p.columns if c.startswith("actual")]), on=["time_h1", "time_hL", "node"])
    df.insert(0, "dataset", ds); frames.append(df)
samples = pd.concat(frames, ignore_index=True)
models = [c[:-3] for c in samples.columns if c.endswith("_hL") and not c.startswith(("actual", "time"))]

fig = go.Figure()
keys = [(ds, n) for ds, g in samples.groupby("dataset", sort=False) for n in sorted(g.node.unique())]
for ds, n in keys:
    d = samples.query("dataset == @ds and node == @n")
    vis = (ds, n) == keys[0]
    fig.add_scatter(x=d.time_hL, y=d.actual_hL, name="actual", line=dict(color="black", width=2), visible=vis)
    for m in models:
        fig.add_scatter(x=d.time_hL, y=d[f"{m}_hL"], name=m, visible=vis, opacity=0.85)
n_tr = 1 + len(models)
fig.update_layout(title="Longest-horizon forecast vs. actual (choose dataset / sensor)", height=450,
                  updatemenus=[dict(active=0, x=0, xanchor="left", y=1.15, buttons=[
                      dict(label=f"{ds} · node {n}", method="update",
                           args=[{"visible": [(k == i) for k in range(len(keys)) for _ in range(n_tr)]}]) for i, (ds, n) in enumerate(keys)])])
fig.show()

## 4. Training curves

In [ ]:
curves = pd.read_csv(RESULTS / "train_curves_pdformer.csv")
fig = px.line(curves, x="epoch", y="val_MAE", color="model", facet_col="dataset", markers=True, title="Validation MAE per epoch")
fig.update_yaxes(matches=None, showticklabels=True); fig.update_xaxes(matches=None)
fig.show()

## 5. Propagation delay: evidence and ablation

In [ ]:
lag = pd.read_csv(RESULTS / "lag_summary.csv").set_index("dataset")
abl = pd.read_csv(RESULTS / "ablation_delay.csv").set_index("dataset") if (RESULTS / "ablation_delay.csv").exists() else pd.DataFrame()
eff = pd.read_csv(RESULTS / "efficiency_pdformer.csv")
display(lag.round(3)); display(abl.round(3)); display(eff.round(2))

**Reading the dashboard**

- Section 1–2: PDFormer has the lowest error everywhere and the flattest horizon curve; the traditional
  models are competitive only for the first one or two steps.
- Section 3: the transformer tracks the rush-hour ramps an hour ahead; Historical Average draws the
  "usual day", Persistence lags reality by a full horizon.
- Section 5: the share of graph edges whose correlation peaks off lag 0 (highways: substantial; taxi
  grid: ~none) lines up with the delay-module ablation gain — see `propagation_delay_findings.md`.